# Добавить пользователей с полным доступом (ГО ДТПП)

Роль `GO_DTPP` + `is_all_filials = '1'`: все листы, все филиалы. Добавляет только строки в `sbx_da.rls_acq_user`; если логин уже есть, его не трогает. Таблицы не пересоздаёт (в отличие от Run All в `rls_acq_roles_build`).

Логин должен совпадать с RAISA: `SELECT '{{ current_username() }}'` в SQL Lab под этим человеком. Регистр не важен.

Запускать сверху вниз под своей учёткой DRP. Если LDAP отказал, не повторяйте больше 2 раз.

In [ ]:
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

NEW_USERS = [
    "fomichev-pv",
    "ilina-val",
    "zaytseva-rva",
]
ROLE = "GO_DTPP"
COMMENT = "ГО, полный доступ: все листы, все РФ"
DRP_USER_DEFAULT = "Shestopalov-VYur"

if "drp" not in globals() or drp is None:
    drp_user = input("DRP user: ").strip() or DRP_USER_DEFAULT
    drp = connect(to="DRP", user_params={"user_name": drp_user, "password": getpass("DRP password: ")})


def fetch(sql):
    with drp:
        df = drp.fetch(sql)
    return pd.DataFrame() if df is None else df


def q(s):
    return str(s).strip().replace("'", "''")


users_in = ", ".join(f"lower('{q(u)}')" for u in NEW_USERS)
USERS_SQL = (
    "SELECT username, role, is_all_filials, comment FROM sbx_da.rls_acq_user "
    f"WHERE lower(BTRIM(username)) IN ({users_in}) ORDER BY 1"
)

display(fetch("SELECT current_user AS me"))
print("Уже есть в справочнике:")
display(fetch(USERS_SQL))

## Добавление

Повторный запуск безопасен: существующие логины пропускаются.

In [ ]:
with drp:
    drp.execute("SET statement_timeout = '30s'")
    for u in NEW_USERS:
        drp.execute(
            "INSERT INTO sbx_da.rls_acq_user (username, role, is_all_filials, comment) "
            f"SELECT '{q(u)}', '{ROLE}', '1', '{q(COMMENT)}' "
            "WHERE NOT EXISTS (SELECT 1 FROM sbx_da.rls_acq_user "
            f"WHERE lower(BTRIM(username)) = lower('{q(u)}'))"
        )
        print("OK:", u)

result = fetch(USERS_SQL)
display(result)

found = set(result["username"].str.strip().str.lower()) if not result.empty else set()
missing = [u for u in NEW_USERS if u.lower() not in found]
wrong = result[(result["role"].str.strip() != ROLE) | (result["is_all_filials"].astype(str).str.strip() != "1")] if not result.empty else result

if missing:
    print("НЕ ДОБАВЛЕНЫ:", missing)
elif not wrong.empty:
    print("Эти логины уже были с другой ролью, тетрадка их не меняла:")
    display(wrong)
else:
    print("ГОТОВО: все", len(NEW_USERS), "с ролью", ROLE, "и всеми филиалами")

## Перевести существующего пользователя на полный доступ

Для тех, кто уже есть в справочнике с ограниченной ролью (РФ КМ, РФ РОЭ, ГО Бизнес). Меняет роль на `GO_DTPP` и `is_all_filials` на `1`, удаляет его строки из `rls_acq_user_filial` (при «все филиалы» они не нужны). Других пользователей не трогает.

Нужна первая ячейка (подключение). Блок «Добавление» для этого запускать не обязательно.

In [ ]:
UPGRADE_USERS = [
    "elina-ad",
]
UPGRADE_COMMENT = "ГО, полный доступ: все листы, все РФ (повышен 23.11.2026)"

up_in = ", ".join(f"lower('{q(u)}')" for u in UPGRADE_USERS)
UP_USERS_SQL = (
    "SELECT username, role, is_all_filials, comment FROM sbx_da.rls_acq_user "
    f"WHERE lower(BTRIM(username)) IN ({up_in}) ORDER BY 1"
)
UP_FILIALS_SQL = (
    "SELECT username, filial_filter FROM sbx_da.rls_acq_user_filial "
    f"WHERE lower(BTRIM(username)) IN ({up_in}) ORDER BY 1, 2"
)

print("До:")
before_up = fetch(UP_USERS_SQL)
display(before_up)
display(fetch(UP_FILIALS_SQL))

missing_up = [u for u in UPGRADE_USERS if before_up.empty or u.lower() not in set(before_up["username"].str.strip().str.lower())]
if missing_up:
    raise RuntimeError(f"Нет в справочнике: {missing_up}. Таких добавляйте через NEW_USERS в первой ячейке.")

with drp:
    drp.execute("SET statement_timeout = '30s'")
    drp.execute(
        f"UPDATE sbx_da.rls_acq_user SET role = '{ROLE}', is_all_filials = '1', comment = '{q(UPGRADE_COMMENT)}' "
        f"WHERE lower(BTRIM(username)) IN ({up_in})"
    )
    drp.execute(f"DELETE FROM sbx_da.rls_acq_user_filial WHERE lower(BTRIM(username)) IN ({up_in})")

print("После:")
after_up = fetch(UP_USERS_SQL)
display(after_up)
left_filials = fetch(UP_FILIALS_SQL)

ok = (
    len(after_up) == len(UPGRADE_USERS)
    and (after_up["role"].str.strip() == ROLE).all()
    and (after_up["is_all_filials"].astype(str).str.strip() == "1").all()
    and left_filials.empty
)
print("ГОТОВО: полный доступ" if ok else "ПРОВЕРЬТЕ таблицы выше, что-то не применилось")